# Track 1, Step 5: Ranking and Submission

This notebook reads the prioritized candidates from Step 4 (`data/track1_step4_candidates.tsv`), applies the EPCR scoring logic from `src/epcr.py`, groups the variants by gene (handling compound-heterozygous pairing), ranks them, and outputs the top 10 candidates to the final submission template.


In [1]:
import sys
import csv
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

from notebooks.utils.epcr import assign_epcr, what_if_table
from notebooks.utils.track1_step5_submission import build_rows, PROBAND_ID

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

### 1. Load Data
We load the TSV containing the variants scored by phenotype.

In [2]:
input_tsv = REPO_ROOT / "data" / "track1_step4_candidates.tsv"

with open(input_tsv, newline="") as fh:
    variants = list(csv.DictReader(fh, delimiter="\t"))

print(f"Loaded {len(variants):,} variants from Step 4.")

Loaded 905 variants from Step 4.


### 2. Build Candidate Rows & EPCR Scoring
Using the logic defined in `notebooks/utils/track1_step5_submission.py` and `src/epcr.py`, we construct the candidate rows. Genes with two or more heterozygous variants are grouped as compound-heterozygous pairs.

In [3]:
max_af_threshold = 0.01
rows = build_rows(variants, chr_prefix=True, max_af=max_af_threshold)
print(f"Built {len(rows):,} candidate rows (genes with >=2 het go as a pair; gnomAD AF > {max_af_threshold} discarded).")

# Rank and assign EPCR, keeping top 10 as specified
max_submission_rows = 10
ranked_rows = assign_epcr(rows, max_rows=max_submission_rows, tail_ratio=0.6, top_cap=0.95)

Built 180 candidate rows (genes with >=2 het go as a pair; gnomAD AF > 0.01 discarded).


### 3. Generate Submission Output
We format the top candidates according to the required Hackathon submission structure.

In [4]:
submission_data = []
for r in ranked_rows:
    v1 = r.variants[0]
    v2 = r.variants[1] if len(r.variants) > 1 else ("", "", "", "")
    submission_data.append({
        "proband_id": PROBAND_ID,
        "chrom_1": v1[0], "pos_1": v1[1], "ref_1": v1[2], "alt_1": v1[3],
        "chrom_2": v2[0], "pos_2": v2[1], "ref_2": v2[2], "alt_2": v2[3],
        "epcr": f"{r.epcr:.4f}", 
        "finding_type": r.finding_type, 
        "notes": r.notes
    })

df_submission = pd.DataFrame(submission_data)
display(df_submission)

out_csv = REPO_ROOT / "output" / "track1" / "lostanos_hpo_matcher_v1.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
df_submission.to_csv(out_csv, index=False)
print(f"\\nSaved df_submission to {out_csv}")

,proband_id,chrom_1,pos_1,ref_1,alt_1,chrom_2,pos_2,ref_2,alt_2,epcr,finding_type,notes
0,PROBAND01,chr15,40209701,T,G,chr15,40220612,T,G,0.9379,primary,compound-het | BUB1B stop_gained HIGH HET pheno=0.63 patho=0.95 inh=1.00 freq=0.83 AVI_PHRED=33.76 MVA-panel | BUB1B missense_variant MODERATE HET pheno=0.63 patho=0.65 inh=1.00 freq=1.00 AVI_PHRED=25.61 MVA-panel
1,PROBAND01,chr22,20996720,C,G,,,,,0.5627,primary,LZTR1 stop_gained HIGH HET pheno=0.41 patho=0.95 inh=0.45 freq=1.00 AVI_PHRED=18.07
2,PROBAND01,chr6,32584174,GCC,G,,,,,0.3376,primary,HLA-DRB1 frameshift_variant HIGH HET pheno=0.49 patho=0.90 inh=1.00 freq=0.24
3,PROBAND01,chr6,32641499,TGG,T,chr6,32641519,A,AG,0.2026,primary,compound-het | HLA-DQA1 frameshift_variant HIGH HET pheno=0.37 patho=0.90 inh=1.00 freq=0.39 | HLA-DQA1 frameshift_variant HIGH HET pheno=0.37 patho=0.90 inh=1.00 freq=0.38
4,PROBAND01,chr6,32641521,CA,C,,,,,0.1216,primary,HLA-DQA1 frameshift_variant HIGH HET pheno=0.37 patho=0.90 inh=1.00 freq=0.37
5,PROBAND01,chr3,48584547,C,T,,,,,0.0730,primary,COL7A1 missense_variant MODERATE HET pheno=0.50 patho=0.62 inh=0.45 freq=1.00 AVI_PHRED=23.82
6,PROBAND01,chr11,17114407,C,A,,,,,0.0438,primary,PIK3C2A missense_variant MODERATE HET pheno=0.54 patho=0.64 inh=0.45 freq=0.76 AVI_PHRED=24.95
7,PROBAND01,chr6,32641501,G,C,,,,,0.0263,primary,HLA-DQA1 missense_variant MODERATE HET pheno=0.37 patho=0.56 inh=1.00 freq=0.71 AVI_PHRED=19.84
8,PROBAND01,chr4,125448901,T,C,,,,,0.0158,primary,FAT4 missense_variant MODERATE HET pheno=0.41 patho=0.67 inh=0.45 freq=0.90 AVI_PHRED=26.54
9,PROBAND01,chr21,45176351,T,C,,,,,0.0095,primary,ADARB1 missense_variant MODERATE HET pheno=0.42 patho=0.59 inh=0.45 freq=1.00 AVI_PHRED=20.16


\nSaved df_submission to /home/samuele/Projects/Learning/rare-disease-real-kid/output/track1/lostanos_hpo_matcher_v1.csv


### 4. What-If Table (Leaderboard Simulation)
This table shows what rank points and F-max the leaderboard would return if row $k$ were the true causal variant.

In [5]:
what_if = what_if_table(ranked_rows)
df_what_if = pd.DataFrame(what_if)
display(df_what_if)

,if_true_row,epcr,rank_points,f_max,f_max_threshold,full_match_rank,partial_match_rank
0,1,0.9379,100.0,1.000,0.9379,1,None
1,2,0.5627,50.0,0.500,0.5627,2,None
2,3,0.3376,50.0,0.400,0.3376,3,None
3,4,0.2026,25.0,0.500,0.2026,4,None
4,5,0.1216,25.0,0.250,0.1216,5,None
5,6,0.0730,10.0,0.222,0.0730,6,None
6,7,0.0438,10.0,0.200,0.0438,7,None
7,8,0.0263,10.0,0.182,0.0263,8,None
8,9,0.0158,10.0,0.167,0.0158,9,None
9,10,0.0095,10.0,0.154,0.0095,10,None
